# Benchmark de modelos ocultos — Classificação tabular pequena (sem treino/tuning)

**Problema:** A small tabular classification dataset: about 400 rows, 15 numeric features, 3 balanced classes, clean. I want strong accuracy with essentially no training time and no hyperparameter tuning. Which model would you use?

**Tipo:** GANHO (oculto que os crus esqueceram)

**Modelo oculto em questão:** TabPFN

## Metodologia

Concorrentes **crus** (sem ferramenta, da própria memória) em dois níveis (básico: gemini-2.5-flash, gpt-5-mini, Claude Sonnet; fronteira: gemini-2.5-pro, gpt-5, Claude Opus). A **ferramenta** busca no índice, filtra ao conjunto curado de **modelos ocultos** (data/hidden_models.json) e reordena por `stat_fit`. Pergunta: o modelo crus lembra do oculto ideal, ou esquece? gemini/openai por API; Sonnet/Opus por subagente.

## Nível básico (crus)

**gemini-2.5-flash**

I would use **k-Nearest Neighbors (kNN)**.

**Why:** It's a lazy learning algorithm, meaning there is essentially no explicit training phase—it simply stores the training data. This perfectly meets the "no training time" requirement. It's a non-parametric method capable of capturing complex decision boundaries and often performs well out-of-the-box with default settings (e.g., `k=5`, Euclidean distance) on clean, balanced datasets, providing strong accuracy without hyperparameter tuning.

**Key Trade-off:** While training is instantaneous, prediction time can be slow for very large datasets or high-dimensional data, as it requires calculating distances to all training samples for each new prediction. Memory usage can also be high as it must store the entire training dataset.

**gpt-5-mini**

Model: RandomForestClassifier (scikit-learn's default RandomForest).

Why: robust out-of-the-box for small tabular data — handles numeric features without scaling, captures nonlinear interactions, rarely needs tuning, and trains essentially instantly on 400 rows.

Key trade-off: less interpretable and sometimes slightly outperformed by a well-tuned gradient-boosted model (e.g., XGBoost/LightGBM) — but those typically require hyperparameter tuning to beat the default RF.

**Claude Sonnet** (claude-sonnet-4-6, subagente)

MODEL: Linear Discriminant Analysis (LDA)

With 400 rows, 15 numeric features and 3 balanced classes, LDA fits analytically in one pass (no training time, no hyperparameters) and is a strong, overfit-resistant bet on small clean tabular data. Trade-off: if the true boundary is nonlinear, a tuned ensemble would beat it.

## Nível fronteira (crus)

**gemini-2.5-pro**

**Linear Discriminant Analysis (LDA)**

*   **Why:** It has an analytical solution, meaning it trains instantly with no iterative optimization. It has no hyperparameters to tune and is a surprisingly powerful classifier that is robust to overfitting on small datasets.

*   **Key Trade-off:** Its core assumption is a linear decision boundary, so it will not capture complex, non-linear relationships between the features.

**gpt-5**

Linear Discriminant Analysis (LDA)

Why: It trains essentially instantly (closed-form), needs no hyperparameter tuning, handles multiclass natively, and is often very accurate on small, clean numeric tabular data by pooling covariance (low variance).

Key trade-off: Assumes roughly Gaussian features with equal class covariances and yields linear decision boundaries; if the true boundary is non-linear, ensembles like Random Forest may do better but at the cost of more training/tuning.

**Claude Opus** (claude-opus-4-8, subagente)

MODEL: Random Forest

Strongest default here: excellent out-of-the-box accuracy on a small clean tabular set with essentially no tuning, robust to scaling, instant to train at this size. Trade-off: black-box vs a transparent model, and a tuned booster might edge it out, but not without the tuning you want to avoid.

## Ferramenta (modelos ocultos que cabem)

Filtrado ao conjunto de ocultos, reordenado por `stat_fit`. Candidatos ocultos recuperados: 2.

| # | modelo oculto | ano | fitScore | razões |
|---|---|---|---|---|
| 1 | TabPFN | 2022 | +9.02 | +target multiclass; +features supported; +n=400 in model sweet spot; +supports zero_training |
| 2 | TimeGPT | 2023 | -7.99 | -target mismatch (model: time-series,continuous); -feature type mismatch (model: temporal); +supports zero_training |

**Oculto-alvo no top-3:** SIM

## Análise imparcial

| Concorrente | Nível | Nomeou |
|---|---|---|
| gemini-2.5-flash | básico | k-NN |
| gpt-5-mini | básico | Random Forest |
| Claude Sonnet | básico | LDA |
| gemini-2.5-pro | fronteira | LDA |
| gpt-5 | fronteira | LDA |
| Claude Opus | fronteira | Random Forest |
| **Ferramenta** | — | **TabPFN** |

**Ganho limpo, e o caso mais claro do conjunto.** Os 6 crus (básico e fronteira) deram clássicos válidos e **nenhum** citou o TabPFN, que é o modelo desenhado exatamente para a restrição (tabular pequeno, sem treino, sem tuning). Só a ferramenta o trouxe. Aqui o modelo oculto é genuinamente esquecido por todos os níveis, e a ferramenta agrega.

## Reprodução

In [ ]:
import bench_lib as B
case = B.case_by_name('classification')
# cru (pago):
print(B.call_gemini(case['prompt'], B.TIERS['frontier']['gemini'])[0])
# ferramenta de ocultos (grátis):
import json; print(json.dumps(B.tool_overlooked(case), indent=2, ensure_ascii=False))